In [1]:
### Batch Rating Curve Evaluation Notebook ###

# Imports
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
from pathlib import Path
from io import StringIO

# Output and display settings (edit as needed)
OUT_DIR = Path("Plots_Discharge")/"Evaluate_Rating_Curves" 
OUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DPI = 150

# Color and palette options
COLORS = {
    "obs_hist": "#1f77b4",    # blue
    "obs_post": "#ff7f0e",    # orange
    "one_to_one": "#990099",  # magenta for 1:1 line
    "rc_line": "#ff00ff"      # RC fit line color
}
PALETTE_MODE = "two-tone"
EVAL_AT = "target_station"

# Date windows you analyze (batch-wide globals)
HIST_START, HIST_END = "2006-01-01", "2014-05-31"
FULL_START, FULL_END = "2006-01-01", "2025-09-01"
SHORT_GAP_DAYS = 3



In [2]:
### All utilities and helpers ###

objs = joblib.load("cache/tributary_cache.joblib")
flow_data_reindexed = objs.get("flow_data_reindexed")
filled_flow_data = objs.get("filled_flow_data")
station_metadata = objs.get("station_metadata")
station_names = objs.get("station_names")
print("Loaded keys:", list(objs.keys()))

def _as_idx(df):
    """Ensure dataframe is indexed by datetime for easy ops."""
    if "Date" in df.columns:
        df = df.copy()
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
        df = df.set_index("Date").sort_index()
    return df

def _f2(x): return f"{x:.2f}" if np.isfinite(x) else "NA"
def _f0(x): return f"{x:.0f}" if np.isfinite(x) else "NA"

def fmt_stats_line(s, label):
    return (
        f"{label}: N={s['N']} r={_f2(s['R'])} R2={_f2(s['R2'])} "
        f"RMSE={_f0(s['RMSE_pct'])}% NSE={_f2(s['NSE'])} "
        f"Bias={_f0(s['Bias_pct'])}%"
    )

def obs_colors():
    hist = COLORS["obs_hist"]
    post = COLORS["obs_post"] if PALETTE_MODE == "two-tone" else COLORS["obs_hist"]
    return hist, post

def axis_minmax(arr, pad_frac=0.03):
    a = np.asarray(arr, float)
    a = a[np.isfinite(a)]
    if a.size == 0: return None, None
    lo, hi = a.min(), a.max()
    span = hi - lo if hi > lo else 1.0  # Avoid collapse
    pad = pad_frac * span
    return lo - pad, hi + pad

def rc_kind(rc_str: str) -> str:
    s = (rc_str or "").strip()
    has_q = re.search(r"\bQ\b", s) is not None
    has_h = re.search(r"\bH\b", s) is not None or re.search(r"\bh\b", s) is not None
    if has_h and not has_q: return "H"
    if has_q and not has_h: return "Q"
    if has_h and has_q:     return "HQ"
    return "unknown"

def kind_title(kind: str) -> str:
    if kind == "H": return "Stage–Discharge Rating (H→Q)"
    if kind == "Q": return "Index‑gage Rating (Q–Q)"
    if kind == "HQ": return "Rating using Auxiliary + Index (H & Q)"
    return "Rating"

def role_tag(kind: str, donor_sid: str, target_sid: str) -> str:
    if donor_sid == (target_sid or donor_sid): return ""
    if kind == "H": return " (Auxiliary gage)"
    if kind == "Q": return " (Index gage)"
    return ""

def _eval_rc(rc_str, H, Q):
    try:
        y = eval(rc_str, {"np": np}, {"H": H, "h": H, "Q": Q})
        y = np.array(y, dtype=float)
    except Exception:
        out = []
        for h, q in zip(H, Q):
            try:
                out.append(float(eval(rc_str, {"np": np}, {"H": float(h), "h": float(h), "Q": float(q)})))
            except Exception:
                out.append(np.nan)
        y = np.array(out, dtype=float)
    y[~np.isfinite(y)] = np.nan
    return y

def compute_extended_stats(obs, pred):
    """
    Compute summary metrics for RC fit:
      - N (sample size)
      - r (Pearson correlation)
      - R2
      - RMSE_pct (root mean square error in percent)
      - NSE (Nash–Sutcliffe efficiency)
      - Bias_pct (percent bias)
    Ignores nan on pairwise basis.
    """
    mask = np.isfinite(obs) & np.isfinite(pred)
    o, p = np.asarray(obs)[mask], np.asarray(pred)[mask]
    if len(o) == 0:
        print("WARNING: No valid data pairs for stats calculation.")
    N = len(o)
    R = np.corrcoef(o, p)[0, 1] if N >= 3 else np.nan
    R2 = R**2 if np.isfinite(R) else np.nan
    RMSE = np.sqrt(np.mean((p - o) ** 2)) if N > 0 else np.nan
    denom = np.mean(np.abs(o)) if N > 0 else np.nan
    RMSE_pct = 100 * RMSE / denom if denom else np.nan
    Bias_pct = 100 * np.mean(p - o) / denom if denom else np.nan
    # Nash–Sutcliffe Efficiency
    ss_res = np.sum((o - p) ** 2)
    ss_tot = np.sum((o - np.mean(o)) ** 2)
    NSE = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan
    return {
        "N": N,
        "R": R,
        "R2": R2,
        "RMSE_pct": RMSE_pct,
        "NSE": NSE,
        "Bias_pct": Bias_pct
    }
    

Loaded keys: ['flow_data', 'flow_data_reindexed', 'filled_flow_data', 'station_metadata', 'station_names']


In [3]:
### Evaluate a station's Rating Curve and generate metrics and plots ###

def review_rc_originals_only(sid, prefer_dv=True, out_dir=OUT_DIR):
    """
    Main batch evaluation for one station:
    - Evaluates the saved rating curve ("RC") for a station using only "original" (not infilled/gapfilled) observed discharge values.
    - Plots:
        * Predicted vs. observed discharge (with 1:1 line, HIST/POST-HIST colors)
        * Predictor variable (H or Q) vs. Q with RC equation overlay
    - Writes QC plots to disk and returns a dict of fit stats/metadata.
    """
    # Load station metadata, RC string, donor station
    meta = station_metadata.get(sid, {})
    rc_str = (meta.get("Rating Curve (Python Format)") or "").strip()
    donor = (meta.get("Station ID used for rating curve") or "").strip() or sid
    if not rc_str or rc_str.lower() in ("na", "linear interpolation"):
        print(f"{sid}: No valid rating curve.")
        return None
    if sid not in filled_flow_data:
        print(f"{sid}: not in filled_flow_data; skipping.")
        return None
    if donor not in flow_data_reindexed:
        print(f"{sid}: donor {donor} not in flow_data_reindexed; skipping.")
        return None

    name = station_names.get(sid, sid)
    kind = rc_kind(rc_str)  # "H", "Q", "HQ", or "unknown"

    # Load and index dataframes for target and donor
    tgt = _as_idx(filled_flow_data[sid].copy())
    don = _as_idx(flow_data_reindexed[donor].copy())

    # Identify original observations for the target
    tgt_method = tgt.get("Discharge_filled_method", pd.Series(index=tgt.index, dtype=object)).astype(str).str.strip().str.lower()
    tgt_original = (tgt_method.eq("") | tgt_method.eq("original")) & tgt["Discharge"].notna()

    has_stage = "Stage" in don.columns

    # --- Build predictors and mask for evaluation (depends on EVAL_AT setting) ---
    if EVAL_AT == "rc_station":
        # Evaluate at donor site
        don_Q_ok = don["Discharge"].notna() if "Discharge" in don.columns else pd.Series(False, index=don.index)
        mask_idx = don_Q_ok
        y_obs = pd.to_numeric(don.loc[mask_idx, "Discharge"], errors="coerce")
        H_pref = pd.to_numeric(don.get("Stage"), errors="coerce").reindex(y_obs.index) if has_stage else pd.Series(index=y_obs.index, dtype=float)
        Q_in   = pd.to_numeric(don.get("Discharge"), errors="coerce").reindex(y_obs.index)
        if has_stage:
            H_pref = H_pref.interpolate(limit=SHORT_GAP_DAYS, limit_direction="both", limit_area="inside")
        x_label_overlay = f"H at {donor}" if ("h" in rc_str.lower()) else f"Q at {donor}"
        y_label_overlay = f"Q at {donor}"
        who_sid = donor
    else:
        # Evaluate at target station using all "original" Discharge values
        mask_idx = tgt_original
        y_obs = pd.to_numeric(tgt.loc[mask_idx, "Discharge"], errors="coerce")

        # Build aligned donor predictors
        if kind == "Q":
            don_method = don.get("Discharge_filled_method", pd.Series(index=don.index, dtype=object)).astype(str).str.strip().str.lower()
            don_original = (don_method.eq("") | don_method.eq("original")) & don["Discharge"].notna()
            overlap_dates = sorted(set(tgt.loc[mask_idx].index) & set(don.loc[don_original].index))
            y_obs = y_obs.reindex(overlap_dates)
            Q_in = pd.to_numeric(don.loc[overlap_dates, "Discharge"], errors="coerce")
            if has_stage:
                H_pref = pd.to_numeric(don.loc[overlap_dates, "Stage"], errors="coerce")
                H_pref = H_pref.interpolate(limit=SHORT_GAP_DAYS, limit_direction="both", limit_area="inside")
            else:
                H_pref = pd.Series(index=y_obs.index, dtype=float)
        else:
            H_pref = pd.to_numeric(don.get("Stage"), errors="coerce").reindex(y_obs.index) if has_stage else pd.Series(index=y_obs.index, dtype=float)
            Q_in   = pd.to_numeric(don.get("Discharge"), errors="coerce").reindex(y_obs.index)
            if has_stage:
                H_pref = H_pref.interpolate(limit=SHORT_GAP_DAYS, limit_direction="both", limit_area="inside")
            Q_in = Q_in.interpolate(limit=SHORT_GAP_DAYS, limit_direction="both", limit_area="inside")
        x_label_overlay = f"H at {donor}" if ("h" in rc_str.lower()) else f"Q at {donor}"
        y_label_overlay = f"Q at {sid}"
        who_sid = sid

    # --- Evaluate the RC for this station ---
    H_arr = H_pref.to_numpy() if isinstance(H_pref, pd.Series) else np.zeros(len(y_obs))
    Q_arr = Q_in.to_numpy()   if isinstance(Q_in, pd.Series) else np.zeros(len(y_obs))
    y_pred_rc = pd.Series(_eval_rc(rc_str, H_arr, Q_arr), index=y_obs.index)

    # --- Create period masks ---
    hist = (y_obs.index >= pd.Timestamp(HIST_START)) & (y_obs.index <= pd.Timestamp(HIST_END))
    full = (y_obs.index >= pd.Timestamp(FULL_START)) & (y_obs.index <= pd.Timestamp(FULL_END))
    post = full & (~hist)

    # --- Compute fit stats ---
    stats_hist_rc = compute_extended_stats(y_obs[hist].to_numpy(), y_pred_rc[hist].to_numpy())
    stats_full_rc = compute_extended_stats(y_obs[full].to_numpy(), y_pred_rc[full].to_numpy())
    title_stats = (
        f"HIST: N={stats_hist_rc['N']} r={stats_hist_rc['R']:.2f} R2={stats_hist_rc['R2']:.2f} "
        f"NSE={stats_hist_rc['NSE']:.2f} | "
        f"FULL: N={stats_full_rc['N']} r={stats_full_rc['R']:.2f} R2={stats_full_rc['R2']:.2f} "
        f"NSE={stats_full_rc['NSE']:.2f}"
    )

    # --- Plot: Predicted vs Observed Q (the "truth") ---
    finite = np.isfinite(y_obs.to_numpy()) & np.isfinite(y_pred_rc.to_numpy())
    if finite.sum() >= 3:
        both_vals = np.concatenate([y_obs.to_numpy()[finite], y_pred_rc.to_numpy()[finite]])
        lo, hi = axis_minmax(both_vals, pad_frac=0.03)
        fig1 = plt.figure(figsize=(10, 7.2))
        plt.axline((0, 0), slope=1, color=COLORS["one_to_one"], lw=1, alpha=0.5, label="1:1")
        fmask = pd.Series(finite, index=y_obs.index)
        hist_color, post_color = obs_colors()
        plt.scatter(y_obs[hist & fmask], y_pred_rc[hist & fmask], s=20, marker="+", c=hist_color, alpha=0.85, label="HIST")
        plt.scatter(y_obs[post & fmask], y_pred_rc[post & fmask], s=20, marker="+", c=post_color, alpha=0.85, label="POST-HIST")
        if lo is not None and hi is not None:
            plt.xlim(lo, hi); plt.ylim(lo, hi)
        who_label = f"Q at {who_sid}"
        plt.xlabel(f"{who_label} (Observed)")
        plt.ylabel(f"{who_label} (RC Predicted)")
        who_title = station_names.get(who_sid, who_sid)
        plt.title(f"RC Predicted vs Observed — {who_title}\n{title_stats}")
        plt.legend(loc="upper left")
        plt.tight_layout()
        fn = f"{sid}_{name}_{'RCStation' if EVAL_AT == 'rc_station' else 'Target'}_RC_vs_Obs.png"
        fig1.savefig(out_dir / fn, dpi=PLOT_DPI, bbox_inches="tight")
        plt.close(fig1)

    # --- Plot: Predictor vs Observed Q with RC curve (shows RC vs. cloud) ---
    if kind in ("H", "Q", "HQ"):
        # Select predictor axis (Stage or Q)
        if kind == "H" and has_stage:
            x_series = H_pref; rc_line_mode = "H"
        elif kind == "Q":
            x_series = Q_in; rc_line_mode = "Q"
        else:  # HQ or fallback
            if has_stage:
                x_series = H_pref; rc_line_mode = "H"
            else:
                x_series = Q_in; rc_line_mode = "Q"

        if rc_line_mode == "H":
            x_label_overlay = f"H at {donor}"
        else:
            x_label_overlay = f"Q at {donor}"

        fmask_pairs = x_series.notna() & y_obs.notna() & y_pred_rc.notna()
        n_pairs = int(fmask_pairs.sum())
        if n_pairs >= 3:
            x_f = x_series[fmask_pairs].to_numpy()
            qobs_f = y_obs[fmask_pairs].to_numpy()
            x_lo, x_hi = axis_minmax(x_f, pad_frac=0.03)
            y_lo, y_hi = axis_minmax(qobs_f, pad_frac=0.03)
            fig2, ax = plt.subplots(figsize=(10, 7.2))
            hist_color, post_color = obs_colors()
            ax.plot(x_series[hist & fmask_pairs], y_obs[hist & fmask_pairs],
                    linestyle="none", marker="+", markersize=4.5, color=hist_color, label="HIST")
            ax.plot(x_series[post & fmask_pairs], y_obs[post & fmask_pairs],
                    linestyle="none", marker="+", markersize=4.5, color=post_color, label="POST-HIST")

            x_grid = np.linspace(np.nanmin(x_f), np.nanmax(x_f), 300)
            # Evaluate RC on x_grid
            if rc_line_mode == "H":
                rc_line = _eval_rc(rc_str, x_grid, np.zeros_like(x_grid))
            else:
                rc_line = _eval_rc(rc_str, np.zeros_like(x_grid), x_grid)
            rc_label = f"RC: {rc_str.replace('**', '^')}"
            ax.plot(x_grid, rc_line, color=COLORS["rc_line"], linewidth=2.0, label=rc_label)
            if x_lo is not None and x_hi is not None: ax.set_xlim(x_lo, x_hi)
            if y_lo is not None and y_hi is not None: ax.set_ylim(y_lo, y_hi)
            ax.set_xlabel(x_label_overlay); ax.set_ylabel(y_label_overlay)
            who_title = station_names.get(sid, sid) if EVAL_AT == "target_station" else station_names.get(donor, donor)
            kind_map = {"H": "Stage–Discharge Rating (H→Q)",
                        "Q": "Index‑gage Rating (Q–Q)",
                        "HQ": "Auxiliary + Index (H & Q)"}
            ax.set_title(f"{kind_map.get(kind, 'Rating')} — {who_title}\n{title_stats}")
            ax.legend(loc="upper left")
            fig2.tight_layout()
            suffix_base = "H_with_RC" if rc_line_mode == "H" else "Q_with_RC"
            suffix = ("RCStation" if EVAL_AT == "rc_station" else "Target") + f"_{suffix_base}.png"
            fig2.savefig(out_dir / f"{sid}_{name}_{suffix}", dpi=PLOT_DPI, bbox_inches="tight")
            plt.close(fig2)

    # --- Return summary metrics for table export ---
    return {
        "Station ID": sid,
        "Station": name,
        "Donor": donor,
        "RC": rc_str,
        "Kind": kind,
        "eval_at": EVAL_AT,
        "N_hist_RC": stats_hist_rc["N"],
        "R_hist_RC": stats_hist_rc["R"],
        "R2_hist_RC": stats_hist_rc["R2"],
        "RMSE_pct_hist_RC": stats_hist_rc["RMSE_pct"],
        "NSE_hist_RC": stats_hist_rc["NSE"],
        "Bias_pct_hist_RC": stats_hist_rc["Bias_pct"],
        "N_full_RC": stats_full_rc["N"],
        "R_full_RC": stats_full_rc["R"],
        "R2_full_RC": stats_full_rc["R2"],
        "RMSE_pct_full_RC": stats_full_rc["RMSE_pct"],
        "NSE_full_RC": stats_full_rc["NSE"],
        "Bias_pct_full_RC": stats_full_rc["Bias_pct"],
        "prefer_dv": prefer_dv
    }

In [4]:
### Find all RC stations with valid RCs and data ###
def is_valid_rc(s):
    f = (s or "").strip().lower()
    return bool(f) and f not in ("na", "linear interpolation")

rc_station_ids = [
    sid for sid, m in station_metadata.items()
    if is_valid_rc(m.get("Rating Curve (Python Format)")) and (sid in filled_flow_data)
]
print(f"Found {len(rc_station_ids)} rating curve station(s) with data.")

Found 11 rating curve station(s) with data.


In [5]:
### Evaluate all RCs and collect metrics ###

results, failures = [], []
for sid in rc_station_ids:
    print(f"Evaluating {sid} — {station_names.get(sid, sid)}")
    try:
        res = review_rc_originals_only(sid, prefer_dv=False, out_dir=OUT_DIR)
        if res: results.append(res)
    except Exception as e:
        print(f"  Error: {e}")
        failures.append({"Station ID": sid, "Error": str(e)})

if not results:
    raise RuntimeError("No results produced by evaluation. Double-check that your data and helpers are loaded.")

Evaluating 07381490 — Atchafalaya River at Simmesport, LA
Evaluating 02470629 — Mobile River at River Mile 31 at Bucks, AL
Evaluating 02471019 — Tensaw River near Mount Vernon, AL
Evaluating 07381000 — Bayou Lafourche at Thibodeaux, LA
Evaluating 07381235 — GIWW West of Bayou Lafourche at Larose, LA
Evaluating 07385790 — Charenton Drainage Canal at Baldwin, LA
Evaluating 07386980 — Vermilion River at Perry, LA
Evaluating 08012150 — Mermentau River at Mermentau, LA
Evaluating 08012470 — Bayou Lacassine near Lake Arthur, LA


C:\Users\p00278065\AppData\Local\anaconda3\Lib\site-packages\numpy\_core\fromnumeric.py:3904: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
C:\Users\p00278065\AppData\Local\anaconda3\Lib\site-packages\numpy\_core\_methods.py:147: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


Evaluating 08015500 — Calcasieu River near Kinder, LA
Evaluating 08041780 — Neches River at Beaumont, TX


In [6]:
### Post-processing, save summary CSV file ###

# Build DataFrame from batch evaluation results
df = pd.DataFrame(results)

# Compute additional correlation columns and add them
def originals_daily_Q(df: pd.DataFrame) -> pd.Series:
    """Get the originals-only, daily mean discharge for a site."""
    df = _as_idx(df)
    method = df.get("Discharge_filled_method", pd.Series(index=df.index, dtype=object)).astype(str).str.strip().str.lower()
    original = (method.eq("") | method.eq("original")) & df.get("Discharge").notna()
    s = pd.to_numeric(df.loc[original, "Discharge"], errors="coerce")
    s = s.groupby(s.index.normalize()).mean()
    # Return for entire period
    return s.reindex(pd.date_range(FULL_START, FULL_END, freq="D"))

def donor_daily_series(df: pd.DataFrame, kind: str) -> pd.Series:
    """Get the relevant donor series: H or Q, originals-only and daily mean."""
    df = _as_idx(df)
    idx = pd.date_range(FULL_START, FULL_END, freq="D")
    if kind == "H":
        if "Stage" not in df.columns:
            return pd.Series(index=idx, dtype=float)
        h = pd.to_numeric(df["Stage"], errors="coerce").groupby(df.index.normalize()).mean()
        return h.reindex(idx)
    else:
        # Q
        if "Discharge" not in df.columns:
            return pd.Series(index=idx, dtype=float)
        method = df.get("Discharge_filled_method", pd.Series(index=df.index, dtype=object)).astype(str).str.strip().str.lower()
        original = (method.eq("") | method.eq("original")) & df["Discharge"].notna()
        qd = pd.to_numeric(df.loc[original, "Discharge"], errors="coerce")
        qd = qd.groupby(qd.index.normalize()).mean()
        return qd.reindex(idx)

def pearson_r(a: pd.Series, b: pd.Series, start: str, end: str):
    """Pearson r over a date range, originals only."""
    a = a.loc[start:end]
    b = b.loc[start:end]
    m = a.notna() & b.notna()
    if m.sum() < 3: return np.nan
    return float(np.corrcoef(a[m].to_numpy(), b[m].to_numpy())[0,1])

def infer_corr_type(rc_str: str) -> str:
    s = (rc_str or "").lower()
    has_h = "h" in s
    has_q = "q" in s
    if has_h and not has_q: return "Q-H"
    if has_q and not has_h: return "Q-Q"
    if has_h and has_q:     return "Q-H"
    return "--"

# Add correlation and legacy columns
R_hist_vals, R_full_vals, corr_types, corr_stations = [], [], [], []

for _, row in df.iterrows():
    sid = str(row["Station ID"]).strip()
    donor = str(row.get("Donor", "")).strip() or sid
    rc_str = (station_metadata.get(sid, {}) or {}).get("Rating Curve (Python Format)", "")
    ctype = infer_corr_type(rc_str)
    corr_types.append(ctype)
    corr_stations.append(donor if ctype in {"Q-H", "Q-Q"} else "")
    if ctype == "--" or sid not in filled_flow_data or donor not in flow_data_reindexed:
        R_hist_vals.append(np.nan); R_full_vals.append(np.nan); continue
    Q_target = originals_daily_Q(filled_flow_data[sid])
    donor_series = donor_daily_series(flow_data_reindexed[donor], "H" if ctype == "Q-H" else "Q")
    R_hist_vals.append(pearson_r(Q_target, donor_series, HIST_START, HIST_END))
    R_full_vals.append(pearson_r(Q_target, donor_series, FULL_START, FULL_END))

df["Correlation Type (from RC)"] = corr_types
df["Correlation Station"] = corr_stations
df["R (HIST)"] = R_hist_vals
df["R (FULL)"] = R_full_vals

# Inline legacy R (2017 CMP)
legacy_csv_text = """Station ID,R (2017 CMP)
03045,1.00
02470629,0.923
02471019,1.00
02479000,0.82
07381000,0.46
07381235,0.63
07381670,0.97
07385790,0.684
07386980,0.49
08012150,0.83
08012470,0.69
08041780,0.90
"""
df_legacy = pd.read_csv(StringIO(legacy_csv_text), dtype={"Station ID": str})
df = df.merge(df_legacy, on="Station ID", how="left")

# Rearrangement for human-friendly CSV
front = [
    "Station ID","Station","Donor","RC","Kind","eval_at",
    "N_hist_RC","R_hist_RC","R2_hist_RC","RMSE_pct_hist_RC","NSE_hist_RC","Bias_pct_hist_RC",
    "N_full_RC","R_full_RC","R2_full_RC","RMSE_pct_full_RC","NSE_full_RC","Bias_pct_full_RC",
    "Correlation Type (from RC)","Correlation Station","R (HIST)","R (FULL)","R (2017 CMP)"
]
other_cols = [c for c in df.columns if c not in front]
cols = [c for c in front if c in df.columns] + other_cols
df_out = df[cols]

# Write to CSV
OUT_CSV = OUT_DIR / "RC_Evaluation_Summary_Combined.csv"
df_out.to_csv(OUT_CSV, index=False)
print("Wrote combined summary CSV:", OUT_CSV)
print(f"Stations evaluated: {len(df_out)}. Failures: {len(failures)}")
if failures:
    print("Failures (first 5):")
    display(pd.DataFrame(failures).head(5))

Wrote combined summary CSV: Plots_Discharge\Evaluate_Rating_Curves\RC_Evaluation_Summary_Combined.csv
Stations evaluated: 11. Failures: 0
